# Chapter 4. Generating Cypher Queries from Natural Language Questions

## The Basics of Query Language Generation

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from utils.utils import chat, neo4j_driver
from utils.schema_utils import get_schema

## The Basics of Query Language Generation

The workfloow of generating Cypher queries from natural language questions typically involves several key steps:
- Retrieve the question from the user.
- Retrieve the schema of the knowledge grahh.
- Define other useful information, such as terminology mappings, format instructions, and few-shot examples.
- Generate the prompt for the language model.
- Pass the prompt to the language model to generate the Cypher query.

## Useful Practices for Query Language Generation

### Using Few-Shot Examples for In-Context Learning

The few-shot examples are specific to the knowledge graph being queried, so they need to be created manually for each knowledge graph. This is useful when we recognize that the LLM misinterprets the schema or makes the same type of mistake.

For example, if the LLM cannot read the country of production of a movie correctly, we can add a few-shot example that shows how to query the country of production for a movie:
```
In what country was the movie The Matrix produced?
Examples
Question: In what country was the movie Ready Player One produced?
Cypher: MATCH (m:Movie { title: 'Ready Player One' })-[:PRODUCED_IN]→(c:Country)
RETURN c.name
```

### Using Database Schema in the Prompt to Show the LLM the Structure of the Knowledge Graph

The schema of the knowledge graph is crucial for generating correct Cypher queries.

The schema should be part of the prompt and make a clear case about what labels,
relationship types, and properties are available in the graph:
```
Graph database schema:
Use only the provided relationship types and properties in the schema. Do not use
any other relationship types or properties that are not provided in the schema.
Node labels and properties:
LabelA {property_a: STRING}
Relationship types and properties:
REL_TYPE {rel_prop: STRING}
The relationships:
(:LabelA)-[:REL_TYPE]->(:LabelB)
(:LabelA)-[:REL_TYPE]->(:LabelC)
```

To infer the schema from Neo4j, we can use the following Cypher query:

```python
NODE_PROPERTIES_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE NOT type = "RELATIONSHIP" AND elementType = "node"
WITH label AS nodeLabels, collect({property:property, type:type}) AS properties
RETURN {labels: nodeLabels, properties: properties} AS output
"""

REL_PROPERTIES_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE NOT type = "RELATIONSHIP" AND elementType = "relationship"
WITH label AS relType, collect({property:property, type:type}) AS properties
RETURN {type: relType, properties: properties} AS output
"""

REL_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE type = "RELATIONSHIP" AND elementType = "node"
UNWIND other AS other_node
RETURN {start: label, type: property, end: toString(other_node)} AS output
"""
```

Then we can get the schema of the graph database and use it in the prompt to the LLM:
```python
def get_structured_schema(driver: neo4j.Driver) -> dict[str, Any]:
    node_labels_response = driver.execute_query(NODE_PROPERTIES_QUERY)
    node_properties = [
        data["output"] for data in [r.data() for r in node_labels_response.records]
    ]

    rel_properties_query_response = driver.execute_query(REL_PROPERTIES_QUERY)
    rel_properties = [
        data["output"]
        for data in [r.data() for r in rel_properties_query_response.records]
    ]

    rel_query_response = driver.execute_query(REL_QUERY)
    relationships = [
        data["output"] for data in [r.data() for r in rel_query_response.records]
    ]

    return {
        "node_props": {el["labels"]: el["properties"] for el in node_properties},
        "rel_props": {el["type"]: el["properties"] for el in rel_properties},
        "relationships": relationships,
    }
```

With this structured response in place, we can format the schema string as we want. We will use a function to format the schema string for the prompt:
```python
def get_schema(driver: neo4j.Driver,) -> str:
    structured_schema = get_structured_schema(driver)

    def _format_props(props: list[dict[str, Any]]) -> str:
        return ", ".join([f"{prop['property']}: {prop['type']}" for prop in props])

    formatted_node_props = [
        f"{label} {{{_format_props(props)}}}"
        for label, props in structured_schema["node_props"].items()
    ]

    formatted_rel_props = [
        f"{rel_type} {{{_format_props(props)}}}"
        for rel_type, props in structured_schema["rel_props"].items()
    ]

    formatted_rels = [
        f"(:{element['start']})-[:{element['type']}]->(:{element['end']})"
        for element in structured_schema["relationships"]
    ]

    return "\n".join(
        [
            "Node properties:",
            "\n".join(formatted_node_props),
            "Relationship properties:",
            "\n".join(formatted_rel_props),
            "The relationships:",
            "\n".join(formatted_rels),
        ]
    )
```

All of the above information can be found under the `schema_utils.py` file in the `utils` directory.

### Adding Terminology Mappings to Semantically Map the User Question to the Schema

The LLM needs to know how to map the terminology used in the question to the terminology used in the schema. A well-designed graph schema uses nouns and verbs for labels and relationship types and adjectives and nouns for properties.

### Format Instructions

Different LLMs output the response in different formats. We can add format instructions to the prompt to make sure that the LLM outputs the Cypher query in the correct format.

## Implementing a Text2Cypher Generator Using a Base Model

We will use the `Movie` graph database for the implementation of the Text2Cypher generator. The `Movie` graph database contains information about movies, actors, directors, and other related entities. We will use the `Movie` graph database to demonstrate how to implement a Text2Cypher generator using a base model.

In [3]:
prompt_template = """
Instructions: 
Generate Cypher statement to query a graph database to get the data to answer the user question below.

Graph Database Schema:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided in the schema.
{schema}

Terminology mapping:
This section is helpful to map terminology between the user question and the graph database schema.
{terminology}

Examples:
The following examples provide useful patterns for querying the graph database.
{examples}

Format instructions:
Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to 
construct a Cypher statement.
Do not include any text except the generated Cypher statement.
ONLY RESPOND WITH CYPHER, NO CODEBLOCKS.

User question: {question}
"""

If we are using Neo4j Desktop, we can install APOC as it can be installed with a few clicks:
- Open Neo4j Desktop and go to your Project.
- Click on the database you are working with (make sure it is stopped).
- On the right-hand side, look for the Plugins tab.
- Expand the APOC section and click Install.
- Restart your database.

In [9]:
schema_string = get_schema(neo4j_driver)
print(schema_string)

Node properties:
Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING, budget: INTEGER}
Genre {name: STRING}
User {userId: STRING, name: STRING}
Actor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}
Director {url: STRING, bornIn: STRING, born: DATE, died: DATE, tmdbId: STRING, imdbId: STRING, name: STRING, poster: STRING, bio: STRING}
Person {url: STRING, died: DATE, bornIn: STRING, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING, bio: STRING}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}
DIRECTED {role: STRING}
The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:RATED]->(:Movie)
(:Actor)-[:ACTED_IN]->(:Movie)
(:Actor)-[:DIRECTED]-

Next we need to write prompts for the terminalogy mappings, format instructions, and few-shot examples.

In [10]:
terminology_string = """
Persons: When a user asks about a person by trade like actor, writer, director, producer, reviewer, they are referring to a node with the label 'Person'.
Movies: When a user asks about a film or movie, they are referring to a node with the label Movie.
"""

examples = [[
    "Who are the two people acted in most movies together?", 
    "MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person) WHERE p1 <> p2 RETURN p1.name, p2.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1"
]]

question = "Who directed the most movies?"

formatted_examples = "\n".join([f"Question: {e[0]}\nCypher: {e[1]}" for i, e in enumerate(examples)])

In [11]:
# The full prompt
full_prompt = prompt_template.format(
    question=question,
    schema=schema_string,
    terminology=terminology_string,
    examples=formatted_examples
)
print(full_prompt)


Instructions: 
Generate Cypher statement to query a graph database to get the data to answer the user question below.

Graph Database Schema:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided in the schema.
Node properties:
Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING, budget: INTEGER}
Genre {name: STRING}
User {userId: STRING, name: STRING}
Actor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}
Director {url: STRING, bornIn: STRING, born: DATE, died: DATE, tmdbId: STRING, imdbId: STRING, name: STRING, poster: STRING, bio: STRING}
Person {url: STRING, died: DATE, bornIn: STRING, born: DATE, imdbId: STRING

With this prompt, we can now generate the Cypher query for the user's question.

In [12]:
cypher = chat(messages=[{"role": "user", "content": full_prompt}])
print(cypher)

MATCH (p:Person)-[:DIRECTED]->(m:Movie)
RETURN p.name, COUNT(m) AS movieCount
ORDER BY movieCount DESC
LIMIT 1


## Specialized (Finetuned) LLMs for Text2Cypher

Training data on HuggingFace: https://huggingface.co/datasets/neo4j/text2cypher-2024v1

Finetuned Models: https://huggingface.co/neo4j